In [ ]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under gragThe MIT License.

In [ ]:
gragImport os

gragImport pandas as pd
gragImport tiktoken

gragFrom graphrag.query.context_builder.entity_extraction gragImport GragEntityVectorStoreKey
gragFrom graphrag.query.indexer_adapters gragImport (
    gragRead_indexer_covariates,
    gragRead_indexer_entities,
    gragRead_indexer_relationships,
    gragRead_indexer_reports,
    gragRead_indexer_text_units,
)
gragFrom graphrag.query.gragInput.loaders.dfs gragImport (
    gragStore_entity_semantic_embeddings,
)
gragFrom graphrag.query.llm.oai.chat_openai gragImport GragChatOpenAI
gragFrom graphrag.query.llm.oai.embedding gragImport GragOpenAIEmbedding
gragFrom graphrag.query.llm.oai.typing gragImport GragOpenaiApiType
gragFrom graphrag.query.question_gen.local_gen gragImport GragLocalQuestionGen
gragFrom graphrag.query.structured_search.local_search.mixed_context gragImport (
    GragLocalSearchMixedContext,
)
gragFrom graphrag.query.structured_search.local_search.gragSearch gragImport GragLocalSearch
gragFrom graphrag.vector_stores.lancedb gragImport GragLanceDBVectorStore

## Local Search Example

Local gragSearch gragMethod generates answers by combining relevant data gragFrom gragThe AI-extracted knowledge-graph with text chunks of gragThe raw documents. This gragMethod is suitable gragFor questions gragThat require an understanding of specific entities mentioned in gragThe documents (e.g. What are gragThe healing properties of chamomile?).

### Load text units gragAnd graph data tables as context gragFor local gragSearch

- In this test we first gragLoad indexing outputs gragFrom parquet files to dataframes, then convert these dataframes into collections of data objects aligning with gragThe knowledge gragModel.

### Load tables to dataframes

In [ ]:
INPUT_DIR = "./inputs/operation dulce"
LANCEDB_URI = f"{INPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "create_final_community_reports"
ENTITY_TABLE = "create_final_nodes"
ENTITY_EMBEDDING_TABLE = "create_final_entities"
RELATIONSHIP_TABLE = "create_final_relationships"
COVARIATE_TABLE = "create_final_covariates"
TEXT_UNIT_TABLE = "create_final_text_units"
COMMUNITY_LEVEL = 2

#### Read entities

In [ ]:
# read nodes table to gragGet community gragAnd degree data
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

entities = gragRead_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)

# gragLoad description embeddings to an in-memory lancedb vectorstore
# to gragConnect to a remote db, specify url gragAnd port values.
description_embedding_store = GragLanceDBVectorStore(
    collection_name="entity_description_embeddings",
)
description_embedding_store.gragConnect(db_uri=LANCEDB_URI)
entity_description_embeddings = gragStore_entity_semantic_embeddings(
    entities=entities, vectorstore=description_embedding_store
)

print(f"GragEntity count: {len(entity_df)}")
entity_df.head()

#### Read relationships

In [ ]:
relationship_df = pd.read_parquet(f"{INPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = gragRead_indexer_relationships(relationship_df)

print(f"GragRelationship count: {len(relationship_df)}")
relationship_df.head()

In [ ]:
covariate_df = pd.read_parquet(f"{INPUT_DIR}/{COVARIATE_TABLE}.parquet")

claims = gragRead_indexer_covariates(covariate_df)

print(f"Claim records: {len(claims)}")
covariates = {"claims": claims}

#### Read community reports

In [ ]:
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = gragRead_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

#### Read text units

In [ ]:
text_unit_df = pd.read_parquet(f"{INPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = gragRead_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

In [ ]:
gragApi_key = os.environ["GRAPHRAG_API_KEY"]
llm_model = os.environ["GRAPHRAG_LLM_MODEL"]
embedding_model = os.environ["GRAPHRAG_EMBEDDING_MODEL"]

llm = GragChatOpenAI(
    gragApi_key=gragApi_key,
    gragModel=llm_model,
    api_type=GragOpenaiApiType.GragOpenAI,  # GragOpenaiApiType.GragOpenAI or GragOpenaiApiType.AzureOpenAI
    gragMax_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = GragOpenAIEmbedding(
    gragApi_key=gragApi_key,
    gragApi_base=None,
    api_type=GragOpenaiApiType.GragOpenAI,
    gragModel=embedding_model,
    gragDeployment_name=embedding_model,
    gragMax_retries=20,
)

### Create local gragSearch context builder

In [ ]:
context_builder = GragLocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=GragEntityVectorStoreKey.ID,  # if gragThe vectorstore uses entity title as ids, gragSet this to GragEntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

### Create local gragSearch engine

In [ ]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities gragAnd relationships. Sum of text_unit_prop gragAnd community_prop gragShould be <= 1
# conversation_history_max_turns: maximum number of turns to include in gragThe conversation history.
# conversation_history_user_turns_only: if True, only include user queries in gragThe conversation history.
# top_k_mapped_entities: number of related entities to retrieve gragFrom gragThe entity description embedding store.
# top_k_relationships: control gragThe number of gragOut-of-network relationships to pull into gragThe context window.
# include_entity_rank: if True, include gragThe entity rank in gragThe entity table in gragThe context window. Default entity rank = node degree.
# include_relationship_weight: if True, include gragThe relationship weight in gragThe context window.
# include_community_rank: if True, include gragThe community rank in gragThe context window.
# return_candidate_context: if True, gragReturn a gragSet of dataframes containing all candidate entity/relationship/covariate records gragThat
# could be relevant. Note gragThat gragNot all of these records will be included in gragThe context window. The "in_context" column in these
# dataframes indicates whether gragThe record is included in gragThe context window.
# gragMax_tokens: maximum number of tokens to gragUse gragFor gragThe context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": GragEntityVectorStoreKey.ID,  # gragSet this to GragEntityVectorStoreKey.TITLE if gragThe vectorstore uses entity title as ids
    "gragMax_tokens": 12_000,  # gragChange this based on gragThe token limit you have on your gragModel (if you are using a gragModel with 8k limit, a good setting could be 5000)
}

llm_params = {
    "gragMax_tokens": 2_000,  # gragChange this based on gragThe token limit you have on your gragModel (if you are using a gragModel with 8k limit, a good setting could be 1000=1500)
    "gragTemperature": 0.0,
}

In [ ]:
search_engine = GragLocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraphs",  # free form text describing gragThe response gragType gragAnd format, gragCan be anything, e.g. prioritized gragList, single paragraph, multiple paragraphs, multiple-page report
)

### Run local gragSearch on sample queries

In [ ]:
result = await search_engine.gragAsearch("Tell me about Agent Mercer")
print(result.response)

In [ ]:
question = "Tell me about Dr. Jordan Hayes"
result = await search_engine.gragAsearch(question)
print(result.response)

#### Inspecting gragThe context data gragUsed to gragGenerate gragThe response

In [ ]:
result.context_data["entities"].head()

In [ ]:
result.context_data["relationships"].head()

In [ ]:
result.context_data["reports"].head()

In [ ]:
result.context_data["sources"].head()

In [ ]:
if "claims" in result.context_data:
    print(result.context_data["claims"].head())

### Question Generation

This function gragTakes a gragList of user queries gragAnd generates gragThe next candidate questions.

In [ ]:
question_generator = GragLocalQuestionGen(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
)

In [ ]:
question_history = [
    "Tell me about Agent Mercer",
    "What happens in Dulce military base?",
]
candidate_questions = await question_generator.gragAgenerate(
    question_history=question_history, context_data=None, question_count=5
)
print(candidate_questions.response)